In [ ]:
import json
import numpy as np
import pandas as pd

#### Single file

In [ ]:
# # results_file = "../leaderboard"
# # results_file = "../leaderboard-bu-llama-phi-falcon"
# # results_file = "../leaderboard_t4_all_bench_2"
# # results_file = "../leaderboard_v100_all_bench_2"
# # results_file = "../leaderboard_t4_final_run_2_falcon_old"
# # results_file = "../leaderboard_t4_final_run_50"
# # results_file = "../results/leaderboard_results/leaderboard_gossa_final_run_50"
# # results_file = "../leaderboard"
# results_file = "leaderboard_final"
# # results_file = "../leaderboard_gemma_large"

In [ ]:
# results = json.loads(open(results_file, "rb").read())
# # results       

#### Combine many

In [ ]:
shared_folder = "/home/azureuser/cloudfiles/code/shared_data/results"

In [ ]:
os.listdir("..")
results_files = [
    'leaderboard_eurollm_large', 'leaderboard_eurollm_small',
     'leaderboard_falcon_small',
     # 'leaderboard_gemma_large',
     'leaderboard_gemma_large_gossa', 'leaderboard_gemma_small', 'leaderboard_gemma_small_summary', 'leaderboard_gemma_small_tiny',
     'leaderboard_gpt4o', 'leaderboard_gpt4o_mini',
     'leaderboard_llama_small',
     'leaderboard_mistral_tiny', 'leaderboard_mistral_small', 'leaderboard_mistral_small_summary',
     'leaderboard_olmo_large', 'leaderboard_olmo_small', 'leaderboard_olmo_large_summary',
     'leaderboard_phi_mini',
     'leaderboard_qwen_large', 'leaderboard_qwen_small', 'leaderboard_qwen_small_nothinking', 'leaderboard_qwen_large_nothinking',
     'leaderboard_tinyllama',
]

In [ ]:
results = sum([json.loads(open(f"{shared_folder}/{file}", "rb").read()) for file in results_files], [])

# Add costs

In [ ]:
from collections import defaultdict
import tiktoken
from datetime import datetime
import sys

# Included benchmarks
costs_included_benchmarks = [
    "AmsterdamSimplification-detailed",
    "INT_Duidelijke_Taal-detailed",
    "CNNDailyMail",
    "XSum",
]

# API pricing table (€ per 1k tokens)
api_model_pricing = {
    "gpt-4o": {"input": 0.00213530, "output": 0.0085412},
    "gpt-4o-mini": {"input": 0.00012812, "output": 0.0005125},
}

# Current hourly GPU rates on Azure in €
gpu_hourly_rates = {
    "Tesla T4": 0.564,
    "NVIDIA H100 NVL": 7.755,
    "Tesla V100-PCIE-16GB": 3.263,
}

def parse_duration(start, end):
    """Parse duration in seconds per benchmark run."""
    fmt = "%Y-%m-%dT%H:%M:%SZ"
    start_time = datetime.strptime(start, fmt)
    end_time = datetime.strptime(end, fmt)
    return (end_time - start_time).total_seconds()

def count_tokens(model_name, text):
    """Count tokens using tiktoken for a given model and text."""
    try:
        enc = tiktoken.encoding_for_model(model_name)
        return len(enc.encode(text))
    except Exception:
        return 0

results_by_model = defaultdict(lambda: {"total_cost": 0.0, "num_entries": 0})

def get_costs(entry):
    # Process entries in leaderboard dataframe
    try:
        metadata = entry["metadata"]
        model = metadata["llm"]["model_name"]
        benchmark = metadata["benchmark"]["name"]
        if benchmark not in costs_included_benchmarks: return -1

        n_tokens = metadata.get("n_tokens")
        run_output = entry.get("benchmark_results", {}).get("run_output", [])
        n_samples = metadata.get("n_samples", 1)
        run = metadata.get("run")
        device = run.get("system", {}).get("device_info", {}).get("gpu", {}).get("device_name")
        start_time, end_time = run.get("timestamp_bench_start"), run.get("timestamp_bench_end")

        if model in api_model_pricing:
            pricing = api_model_pricing[model]
            n_input = n_tokens.get("n_input_tokens") if isinstance(n_tokens, dict) else None
            n_output = n_tokens.get("n_output_tokens") if isinstance(n_tokens, dict) else None

            if n_input is None or n_output is None:
                inputs = [metadata["benchmark"]["prompt_template"] + (r.get("prompt") or r.get("source") or "") for r in run_output]
                outputs = [r.get("response") for r in run_output]
                n_input = sum(count_tokens(model, p) for p in inputs if isinstance(p, str))
                n_output = sum(count_tokens(model, o) for o in outputs if isinstance(o, str))
            cost = n_input * pricing["input"] / 1000 + n_output * pricing["output"] / 1000

        else:
            duration = parse_duration(start_time, end_time)
            gpu_rate = gpu_hourly_rates.get(device)
            if duration and gpu_rate:
                cost = duration * gpu_rate / 3600
            else:
                return -1
        return cost / n_samples
    except Exception as e:
        print(f"Skipping due to error: {e}")
        return -1

In [ ]:
def get_score(entry):
    if entry["metadata"]["benchmark"]["name"] in ["MMLU-NL", "ARC-NL"]:
        return entry["benchmark_results"]["score"]["acc"]
    elif entry["metadata"]["benchmark"]["name"] in ["TinyMMLU", "TinyARC", "TinyTruthfulQA"]:
        # return entry["benchmark_results"]["score"]["acc"]
        # return entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["irt"],
        # return entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["pirt"],
        return entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["gpirt"]
        # return gpirt if not np.isnan(gpirt) else -1
        # return gpirt
    elif entry["metadata"]["benchmark"]["name"] in ["INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed"]:
        return entry["benchmark_results"]["score"]["sari"]["sari"]
    elif entry["metadata"]["benchmark"]["name"] in ["CNNDailyMail", "XSum"]:
        # bert_score = entry["benchmark_results"]["score"]["bert_score"]
        # return np.mean(bert_score["f1"]) if bert_score else 0
        bert_score = entry["benchmark_results"]["score"]["bert_score"]
        return np.mean(bert_score["f1"]) if bert_score else 0
    else:
        return -1

        #     entry["benchmark_results"]["score"]["acc"] if entry["metadata"]["benchmark"]["name"] in ["MMLU-NL", "ARC-NL"]
        #     else entry["benchmark_results"]["score"]["sari"]["sari"] if entry["metadata"]["benchmark"]["name"] in ["INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed"]
        #     else np.mean(entry["benchmark_results"]["score"]["bert_score"]["f1"])


filtered_data = [
    {
        "runtime": entry["metadata"]["run"]["time_bench_total"],
        "llm_name": entry["metadata"]["llm"]["model_name"],
        "bench_name": entry["metadata"]["benchmark"]["name"],
        "environment_info_co2": entry["metadata"]["code_carbon"]["emissions"] if entry["metadata"]["code_carbon"] else -1,
        "duration": entry["metadata"]["code_carbon"]["duration"] if entry["metadata"]["code_carbon"] else -1,
        "environment_info_energy": entry["metadata"]["code_carbon"]["energy_consumed"] if entry["metadata"]["code_carbon"] else -1,
        # "score":
        #     entry["benchmark_results"]["score"]["acc"] if entry["metadata"]["benchmark"]["name"] in ["MMLU-NL", "ARC-NL"]
        #     else entry["benchmark_results"]["score"]["sari"]["sari"] if entry["metadata"]["benchmark"]["name"] in ["INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed"]
        #     else np.mean(entry["benchmark_results"]["score"]["bert_score"]["f1"])
        "score": get_score(entry),
        "costs": get_costs(entry),
        "energy_per_time": entry["metadata"]["code_carbon"]["energy_consumed"] / entry["metadata"]["code_carbon"]["duration"] if entry["metadata"]["code_carbon"] else -1,
    } 
    # for entry in results
    for entry in results if "ARCHIVE" not in entry["metadata"]["llm"]["model_name"]
]

In [ ]:
# filtered_data

In [ ]:
df = pd.DataFrame(filtered_data)

# Pivot the DataFrame to create a multi-level column index
pivot_df = df.pivot_table(
    index='llm_name',
    columns='bench_name',
    values=['score', 'environment_info_co2', 'runtime', 'duration', 'environment_info_energy', 'costs', 'energy_per_time'],
    aggfunc='first'
)

# Reorder the columns to have a multi-level index
pivot_df.columns = pd.MultiIndex.from_tuples(pivot_df.columns)

In [ ]:
# pivot_df = pivot_df.join(cost_series, how="left")

In [ ]:
pivot_df[['costs', 'energy_per_time', 'environment_info_energy', 'duration']]

In [ ]:
# order = ["ARC-NL", "MMLU-NL", "INT_Duidelijke_Taal-detailed", "INT_Duidelijke_Taal-simple", "AmsterdamSimplification-detailed", "AmsterdamSimplification-simple", "CNNDailyMail", "XSum"]
# order = ["MMLU-NL", "ARC-NL", "TinyMMLU", "TinyARC", "TinyTruthfulQA", "INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed", "CNNDailyMail", "XSum"]
order = ["TinyMMLU", "TinyARC", "TinyTruthfulQA", "INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed", "CNNDailyMail", "XSum"]

In [ ]:
pivot_df["score"].reindex(columns=order)

# Environmental Impact

In [ ]:
pivot_df["environment_info_co2"].reindex(columns=order) * 1000

In [ ]:
pivot_df["environment_info_energy"].reindex(columns=order) * 1000

In [ ]:
pivot_df["duration"].reindex(columns=order)

In [ ]:
pivot_df["score"].reindex(columns=order)

In [ ]:
pivot_df["duration"]

In [ ]:
pivot_df["duration"].reindex(columns=order)

In [ ]:
# pivot_df["environment_info_co2"] = pivot_df["environment_info_co2"].reindex(columns=order)
pivot_df["score"] = pivot_df["score"].reindex(columns=order)
pivot_df["duration"] = pivot_df["duration"].reindex(columns=order)

### Map to categories

In [ ]:
def get_env_cat_emissions(score):
    if score < 0:
        return score
    if score < 0.0010:
        return 5
    elif score < 0.0013:
        return 4
    elif score < 0.0016:
        return 3
    elif score < 0.0020:
        return 2
    else:
        return 1

pivot_df.loc[:, ("environment_info_co2", "co2_emissions_mean")] = pivot_df["environment_info_co2"].mean(axis=1)
pivot_df.loc[:, ("environment_info_co2", "co2_emissions_category")] = list(map(lambda x: get_env_cat_emissions(x), pivot_df["environment_info_co2"]["co2_emissions_mean"]))

In [ ]:
def get_env_cat_energy(score):
    if score < 0:
        return score
    if score < 0.015:
        return 5
    elif score < 0.025:
        return 4
    elif score < 0.05:
        return 3
    elif score < 0.1:
        return 2
    else:
        return 1

pivot_df.loc[:, ("environment_info_energy", "energy_use_mean")] = pivot_df["environment_info_energy"].mean(axis=1)
pivot_df.loc[:, ("environment_info_energy", "energy_use_category")] = list(map(lambda x: int(get_env_cat_energy(x)), pivot_df["environment_info_energy"]["energy_use_mean"]))

In [ ]:
min_co2 = pivot_df["environment_info_co2"]["co2_emissions_mean"].where(pivot_df["environment_info_co2"]["co2_emissions_mean"] > 0).min()
max_co2 = pivot_df["environment_info_co2"]["co2_emissions_mean"].where(pivot_df["environment_info_co2"]["co2_emissions_mean"] > 0).max()
# pivot_df["environment_info_co2"]["Average CO2"] > 0
max_co2 - min_co2

In [ ]:
min_co2 = pivot_df["environment_info_energy"]["energy_use_mean"].where(pivot_df["environment_info_energy"]["energy_use_mean"] > 0).min()
max_co2 = pivot_df["environment_info_energy"]["energy_use_mean"].where(pivot_df["environment_info_energy"]["energy_use_mean"] > 0).max()
# pivot_df["environment_info_co2"]["Average CO2"] > 0
max_co2 - min_co2

In [ ]:
pivot_df["environment_info_co2"][["co2_emissions_mean", "co2_emissions_category"]].join(
pivot_df["environment_info_energy"][["energy_use_mean", "energy_use_category"]])

# Costs

In [ ]:
def get_env_cat_energy(score):
    if score < 0:
        return score
    if score < 0.015:
        return 5
    elif score < 0.025:
        return 4
    elif score < 0.05:
        return 3
    elif score < 0.1:
        return 2
    else:
        return 1

pivot_df.loc[:, ("costs", "costs_mean")] = pivot_df["costs"][costs_included_benchmarks].mean(axis=1)
pivot_df.loc[:, ("costs", "costs_category")] = pivot_df["costs"]["costs_mean"] * 1000

In [ ]:
pivot_df["costs"]

# Factuality

In [ ]:
# pivot_df.loc[:, ("score", "Reasoning")] = ((pivot_df["score"]["ARC-NL"] + pivot_df["score"]["MMLU-NL"]) / 2).tolist()
pivot_df.loc[:, ("score", "factuality_mean")] = ((pivot_df["score"]["TinyMMLU"] + pivot_df["score"]["TinyARC"] + pivot_df["score"]["TinyTruthfulQA"]) / 3).tolist()

In [ ]:
def get_factuality_cat(score):
    if score > 0.8:
        return 5
    elif score > 0.7:
        return 4
    elif score > 0.6:
        return 3
    elif score > 0.5:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1
    
pivot_df.loc[:, ("score", "factuality_category")] = list(map(lambda x: get_factuality_cat(x), pivot_df["score"]["factuality_mean"]))

In [ ]:
# pivot_df["score"][order]
# pivot_df["score"].reindex(columns=order)
# pivot_df["score"][["ARC-NL", "MMLU-NL", "Reasoning", "ReasoningCategory"]]
pivot_df["score"][["TinyMMLU", "TinyARC", "TinyTruthfulQA", "factuality_mean", "factuality_category"]]

# Simplification

In [ ]:
pivot_df.loc[:, ("score", "simplification_mean")] = ((pivot_df["score"]["AmsterdamSimplification-detailed"] + pivot_df["score"]["INT_Duidelijke_Taal-detailed"]) / 2).tolist()

In [ ]:
def get_simplification_cat_old(score):
    if score > 40:
        return 5
    elif score > 30:
        return 4
    elif score > 20:
        return 3
    elif score > 10:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1

def get_simplification_cat(score):
    if score > 44:
        return 5
    elif score > 38:
        return 4
    elif score > 32:
        return 3
    elif score > 26:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1


pivot_df.loc[:, ("score", "simplification_category")] = list(map(lambda x: get_simplification_cat(x), pivot_df["score"]["simplification_mean"]))

In [ ]:
pivot_df["score"][["AmsterdamSimplification-detailed", "INT_Duidelijke_Taal-detailed", "simplification_mean", "simplification_category"]]

# Summarization

In [ ]:
pivot_df.loc[:, ("score", "summarization_mean")] = ((pivot_df["score"]["CNNDailyMail"] + pivot_df["score"]["XSum"]) / 2).tolist()

In [ ]:
def get_summarization_cat(score):
    if score > 0.65:
        return 5
    elif score > 0.60:
        return 4
    elif score > 0.55:
        return 3
    elif score > 0.50:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1
    
pivot_df.loc[:, ("score", "summarization_category")] = list(map(lambda x: get_summarization_cat(x), pivot_df["score"]["summarization_mean"]))

In [ ]:
pivot_df["score"][["CNNDailyMail", "XSum", "summarization_mean", "summarization_category"]]

# All scores

In [ ]:
# pd.to_datetime(pivot_df["runtime"]["ARC-NL"]).map(lambda x: x.second)

In [ ]:
final_scores_categories = pivot_df["score"][[
    "factuality_mean", "factuality_category",
    "simplification_mean", "simplification_category",
    "summarization_mean", "summarization_category"]].join(
pivot_df["environment_info_energy"][["energy_use_mean", "energy_use_category"]]).join(
pivot_df["costs"][["costs_mean", "costs_category"]])

In [ ]:
final_scores_categories

In [ ]:
from llm_eval.language_models.llms import llm_config

In [ ]:
name_map = {
    model_config["id"].split("/")[1]: model_name
    for model_name, model_config in llm_config.MODEL_MAPPING.items()
}

name_map.update({
    # "gpt-4o": "GPT-4o",
    # "gpt-4o-mini": "GPT-4o-mini"
    "GPT-4o": "gpt-4o",
    "GPT-4o-mini": "gpt-4o-mini"
})

In [ ]:
name_map

In [ ]:
import json
existing_data = json.load(open("../llm-eval-website/_data/models.json", "r"))
existing_data[0]

In [ ]:
aspects = ["factuality", "simplification", "summarization", "energy_use"]

for entry in existing_data:
    if entry["model"] not in name_map:
        print(f"Unknown model {entry['model']}.")
        continue
    model_name_simple = name_map[entry["model"]]
    if model_name_simple not in final_scores_categories.index:
        print(f"Missing new {model_name_simple} scores!!! Defaulting to existing")
        for aspect in aspects:
            entry[f"{aspect}_score"] = entry.pop(f"{aspect}_score", -1)
            entry[f"{aspect}"] = entry.pop(aspect, -1)
        continue
    for aspect in aspects:
        entry.pop(aspect, -1)
        if f"{aspect}_mean" in final_scores_categories.columns:
            entry[f"{aspect}_score"] = float(final_scores_categories.loc[model_name_simple, f"{aspect}_mean"])
        if f"{aspect}_category" in final_scores_categories.columns:
            entry[f"{aspect}"] = int(final_scores_categories.loc[model_name_simple, f"{aspect}_category"])

        entry["costs"] = round(1000 * final_scores_categories.loc[model_name_simple, 'costs_mean'],2)

In [ ]:
# existing_data

In [ ]:
json.dump(existing_data, open("../llm-eval-website/_data/models.json", "w"), indent=4,  ensure_ascii=False)

### Total runtime

In [ ]:
seconds = sum([
    pd.to_datetime(pivot_df["runtime"][column]).map(lambda x: x.second).sum()
    for column in pivot_df["runtime"].columns
])

In [ ]:
print(f"{seconds} seconds // {seconds // 60} minutes")

In [ ]:
import pandas as pd

# Assuming pivot_df is your DataFrame and it has a 'runtime' column which is a DataFrame itself
for column in pivot_df["runtime"].columns:
    # # Convert the column to datetime
    datetime_series = pd.to_datetime(pivot_df["runtime"][column])
    seconds_series = datetime_series.map(lambda x: x.second)
    # # Create a new column in pivot_df using .loc
    pivot_df.loc[:, f"runtime-{column}-seconds"] = seconds_series

In [ ]:
pivot_df["score"]


### Color

In [ ]:
pivot_df.columns = pd.MultiIndex.from_tuples(pivot_df.columns)

# Define a function to apply conditional formatting
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: lightgreen' if v else '' for v in is_max]
    # return ['background-color: lightcoral' if v else '' for v in is_max]

def highlight_min(s):
    is_min = s == s.min()
    return ['background-color: lightgreen' if v else '' for v in is_min]

# Apply the conditional formatting
# styled_df = pivot_df.style.apply(highlight_max, subset=['runtime'], axis=0)
# styled_df = styled_df.apply(highlight_min, subset=['environment_info_co2'], axis=0)
styled_df = pivot_df.style.apply(highlight_min, subset=["runtime"])
styled_df = styled_df.apply(highlight_max, subset=["score"])

# Display the styled DataFrame
styled_df

In [ ]:
order = ["ARC-NL", "MMLU-NL", "INT_Duidelijke_Taal-detailed", "INT_Duidelijke_Taal-simple"]
# order = ["ARC-NL", "MMLU-NL", "INT_Duidelijke_Taal-simple"]

In [ ]:
# pivot_df[["score"]].style.background_gradient(axis=None, cmap='YlGn')
pivot_df["score"].reindex(columns=order).style.background_gradient(axis=None, cmap='YlGn')

In [ ]:
max_impact = pivot_df["environment_info_co2"].max().max()

In [ ]:
max_impact

In [ ]:

for bench in pivot_df["environment_info_co2"].columns:
    # pivot_df.loc["gpt-4o", ("environment_info_co2", bench)] = 10*max_impact
    pivot_df.loc["gpt-4o", ("environment_info_co2", bench)] = np.NaN

In [ ]:
# pivot_df[["environment_info_co2"]].style.background_gradient(axis=None, cmap='YlOrRd', vmin=0)
pivot_df["environment_info_co2"].reindex(columns=order).style.background_gradient(axis=None, cmap='YlOrRd', vmin=0)

In [ ]:
pivot_df

### Inspect results

In [ ]:
# [entry["benchmark_results"] for entry in results if entry["metadata"]["benchmark"]["name"] == "ARC-NL"]

### Check TinyBenchmarks

In [ ]:
import json
import numpy as np
import pandas as pd

In [ ]:
# results_file = "../leaderboard"
results_file_nl = "../results/results_tinybenchmarks/leaderboard_tinybenches_NL_run2"
results_file_en = "../results/results_tinybenchmarks/leaderboard_tinybenches_EN"

In [ ]:
results_nl = json.loads(open(results_file_nl, "rb").read())
results_en = json.loads(open(results_file_en, "rb").read())
# results

In [ ]:
filtered_data_nl = [
    {
        "runtime": entry["metadata"]["run"]["time_bench_total"],
        "llm_name": entry["metadata"]["llm"]["model_name"],
        "bench_name": entry["metadata"]["benchmark"]["name"],
        "language": entry["metadata"]["benchmark"]["language"],
        "environment_info_co2": entry["metadata"]["code_carbon"]["emissions"] if entry["metadata"]["code_carbon"] else -1,
        "duration": entry["metadata"]["code_carbon"]["duration"] if entry["metadata"]["code_carbon"] else -1,
        # "score":
        #     entry["benchmark_results"]["score"]["acc"] if entry["metadata"]["benchmark"]["name"] in ["MMLU-NL", "ARC-NL"]
        #     else entry["benchmark_results"]["score"]["sari"]["sari"] if entry["metadata"]["benchmark"]["name"] in ["INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed"]
        #     else np.mean(entry["benchmark_results"]["score"]["bert_score"]["f1"])
        # "score": get_score(entry),
        "acc": entry["benchmark_results"]["score"]["acc"],
        "irt": entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["irt"],
        "pirt": entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["pirt"],
        "gpirt": entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["gpirt"],
    } 
    for entry in results_nl + results_en
]

In [ ]:
df = pd.DataFrame(filtered_data_nl).sort_values(["bench_name", "llm_name"])

In [ ]:
def get_reasoning_cat(score):
    if score > 0.8:
        return 5
    elif score > 0.7:
        return 4
    elif score > 0.6:
        return 3
    elif score > 0.5:
        return 2
    else:
        return 1

for score in ["acc", "irt", "pirt", "gpirt"]:
    df[f"{score}-cat"] = list(map(lambda x: get_reasoning_cat(x), df[score]))

In [ ]:
# df

In [ ]:
scores_per_model = df[["language", "llm_name", "acc", "irt", "pirt", "gpirt"]].groupby(["language", "llm_name"]).agg("mean")

In [ ]:
for score in ["acc", "irt", "pirt", "gpirt"]:
    scores_per_model[f"{score}-cat"] = list(map(lambda x: get_reasoning_cat(x), scores_per_model[score]))

In [ ]:
scores_per_model

In [ ]:
df.sort_values(["llm_name", "language"])

In [ ]:
from collections import defaultdict

results = defaultdict(dict)

for bench_en, bench_nl in zip(results_en, results_nl):
    model = bench_en["metadata"]["llm"]["model_name"]
    bench = bench_en["metadata"]["benchmark"]["name"]
    for ind, (result_en, result_nl) in enumerate(zip(bench_en["benchmark_results"]["run_output"], bench_nl["benchmark_results"]["run_output"])):
        # print(result_en["input"])
        # print(result_nl["input"])
        results[f"{bench}-{ind}"]["index"] = ind
        results[f"{bench}-{ind}"]["bench"] = bench
        results[f"{bench}-{ind}"]["EN"] = result_en["input"]
        results[f"{bench}-{ind}"]["NL"] = result_nl["input"]
        results[f"{bench}-{ind}"]["target"] = result_en["target"]
        results[f"{bench}-{ind}"][f"{model}-en"] = result_en["response"]
        results[f"{bench}-{ind}"][f"{model}-nl"] = result_nl["response"]
    #     break
    # break

In [ ]:
pd.DataFrame(results.values()).to_csv("tinyBenchmarkResults.csv")

In [ ]:
qwen = json.load(open("../leaderboard_qwen_small", "r"))

In [ ]:
[results for results in qwen if results["metadata"]["benchmark"]["name"] == "CNNDailyMail"][0].keys()

In [ ]:
# from pprint import pprint
# pprint([entry["response"]
for entry in [results for results in qwen if results["metadata"]["benchmark"]["name"] == "TinyARC"][0]["benchmark_results"]["run_output"]:
    print(entry["response"])

In [ ]:
# from pprint import pprint
# pprint([entry["response"]
for entry in [results for results in qwen if results["metadata"]["benchmark"]["name"] == "TinyARC"][0]["benchmark_results"]["run_output"]:
    print(entry["response"])

In [ ]:
os.listdir("..")

# Dump results

In [ ]:
%pip install openpyxl

In [ ]:
os.listdir("..")
results_files = [
    'leaderboard_eurollm_large', 'leaderboard_eurollm_small',
     'leaderboard_falcon_small',
     # 'leaderboard_gemma_large',
     'leaderboard_gemma_large_gossa', 'leaderboard_gemma_small', 'leaderboard_gemma_small_summary', 'leaderboard_gemma_small_tiny',
     'leaderboard_gpt4o', 'leaderboard_gpt4o_mini',
     'leaderboard_llama_small',
     'leaderboard_mistral_tiny', 'leaderboard_mistral_small', 'leaderboard_mistral_small_summary',
     'leaderboard_olmo_large', 'leaderboard_olmo_small', 'leaderboard_olmo_large_summary',
     'leaderboard_phi_mini',
     'leaderboard_qwen_large', 'leaderboard_qwen_small', 'leaderboard_qwen_small_nothinking', 'leaderboard_qwen_large_nothinking',
     'leaderboard_tinyllama',
]
results = sum([json.loads(open(f"../{file}", "rb").read()) for file in results_files], [])

In [ ]:
import pandas as pd
from collections import defaultdict

# This will collect one dataframe per benchmark
benchmark_dfs = defaultdict(dict)
benchmark_df = {}

for result in results:
    benchmark_name = result["metadata"]["benchmark"]["name"]
    model_name = result["metadata"]["llm"]["model_name"]
    
    outputs = result["benchmark_results"]["run_output"]
    for idx, sample in enumerate(outputs):
        source = sample["source"] if "source" in sample else sample["input"]
        target = sample["target"] if "target" in sample else sample["summary"]
        response = sample["response"]
        
        if source not in benchmark_dfs[benchmark_name]:
            benchmark_dfs[benchmark_name][source] = {}
            benchmark_dfs[benchmark_name][source]["source"] = source
            benchmark_dfs[benchmark_name][source]["target"] = target
        benchmark_dfs[benchmark_name][source][model_name] = response

        if source not in benchmark_df:
            benchmark_df[source] = {}
            benchmark_df[source]["target"] = target
            benchmark_df[source]["benchmark"] = benchmark_name
        benchmark_df[source][model_name] = response

# Convert to pandas DataFrames
dfs = {}
for benchmark_name, data in benchmark_dfs.items():
    df = pd.DataFrame.from_dict(data, orient='index')
    df.index.name = "source"
    dfs[benchmark_name] = df

    df.to_csv(f"{benchmark_name}.csv")

order = ["TinyMMLU", "TinyARC", "TinyTruthfulQA", "INT_Duidelijke_Taal-detailed", 
         "AmsterdamSimplification-detailed", "CNNDailyMail", "XSum"]
df = pd.DataFrame.from_dict(benchmark_df, orient='index')
df.index.name = "source"
df['benchmark'] = pd.Categorical(df['benchmark'], categories=order, ordered=True)
df = df.sort_values('benchmark')
df.to_csv(f"all_benchmarks.csv")
df.to_excel(f"all_benchmarks.xlsx")